# Network-Based Stratification (NBS) Tutorial
## From Somatic Mutations to Patient Subgroups



---

### What You Will Learn

By the end of this tutorial you will be able to:

1. Download and prepare real cancer genomics data from **cBioPortal**
2. Understand and apply **network propagation** to smooth sparse mutation profiles
3. Cluster patients using **NMF** and **hierarchical consensus clustering**
4. Evaluate cluster stability and biological relevance
5. Generate **Kaplan–Meier survival curves** to test clinical significance

---

### Background

Cancer genomes are heterogeneous — each patient's tumour carries a unique set of somatic mutations. Many of these mutations are rare ('long-tail' mutations), making direct comparison of mutation profiles across patients very noisy. 

**Network propagation** addresses this by 'smoothing' each patient's mutation profile over a protein–protein interaction (PPI) network. Genes that are close in the network to a mutated gene receive elevated scores, capturing the idea that mutations in different genes of the same pathway have similar functional consequences.

This is the core idea behind **Network-Based Stratification (NBS)**, introduced by Hofree et al. (2013, *Nature Methods*).

```
Raw mutations  →  Network propagation  →  Smooth profiles  →  Clustering  →  Subtypes
     (sparse)                                (dense)                          (clinical)
```

---
## Section 0 — Installation & Imports

Run the cell below once to install all required packages.

In [15]:
# Install dependencies
# !pip install networkx scikit-learn lifelines scipy matplotlib seaborn requests pandas numpy
# pyNBS is installed separatelly (README)

In [2]:
import os
import warnings
import requests
import io

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All packages imported successfully.")

✅ All packages imported successfully.


---
## Section 1 — Downloading Data from cBioPortal

### 1.1  Choosing Your Dataset

Go to **https://www.cbioportal.org/datasets** and browse available datasets.  
Good choices for this tutorial (reasonable size, well-annotated survival data):

| Cancer type | Study ID | ~Samples |
|-------------|----------|----------|
| Breast (TCGA) | `brca_tcga` | 1084 |
| Glioblastoma (TCGA) | `gbm_tcga` | 543 |
| Lung Adeno (TCGA) | `luad_tcga` | 566 |
| Colon (TCGA) | `coadread_tcga` | 640 |
| Ovarian (TCGA) | `ov_tcga` | 606 |

> **💡 Task:** Pick any dataset that interests you. Set `STUDY_ID` below.

In [ ]:
# ── CHANGE THIS to your chosen study ──────────────────────────────────────────
STUDY_ID = "coadread_tcga"   # e.g. 'brca_tcga', 'luad_tcga', 'ov_tcga'
# ──────────────────────────────────────────────────────────────────────────────

BASE_URL = "https://www.cbioportal.org/api"

def cbio_get(endpoint, params=None):
    """Helper to call cBioPortal REST API and return a DataFrame."""
    url = f"{BASE_URL}/{endpoint}"
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()

    data = r.json()
    if isinstance(data, dict):
        # single object returned (e.g. /studies/{id})
        return pd.DataFrame([data])
    elif isinstance(data, list):
        return pd.DataFrame(data)
    else:
        # fallback for scalar response
        return pd.DataFrame([{"value": data}])

# Fetch study metadata
study_df = cbio_get(f"studies/{STUDY_ID}")
print(f"Study: {study_df.get('name', [STUDY_ID])[0] if isinstance(study_df, pd.DataFrame) and 'name' in study_df else STUDY_ID}")

# Fetch sample list
samples_df = cbio_get("samples", params={"studyId": STUDY_ID, "pageSize": 10000})
sample_ids = samples_df["sampleId"].tolist()
patient_ids = samples_df["patientId"].tolist()
print(f"Total samples: {len(sample_ids)}")

ValueError: Mixing dicts with non-Series may lead to ambiguous ordering.

### 1.2  Downloading Somatic Mutations

We fetch mutation data via the cBioPortal API. Each row is one mutation in one patient.

In [ ]:
# Fetch mutations (may take 30–60 s for large studies)
print("Fetching mutations — this may take a minute...")

mutation_url = f"{BASE_URL}/mutations"
payload = {
    "sampleIds": sample_ids,
    "studyId": STUDY_ID
}

# Use molecular profile for mutations
profiles_df = cbio_get("molecular-profiles", params={"studyId": STUDY_ID})
mut_profile = profiles_df[profiles_df["molecularAlterationType"] == "MUTATION_EXTENDED"]["molecularProfileId"].values[0]
print(f"Using molecular profile: {mut_profile}")

mut_response = requests.post(
    f"{BASE_URL}/mutations/fetch",
    json={"sampleIds": sample_ids, "studyId": STUDY_ID},
    params={"molecularProfileId": mut_profile, "projection": "SUMMARY"},
    timeout=120
)
mut_response.raise_for_status()
mutations_df = pd.DataFrame(mut_response.json())

print(f"Total mutations fetched: {len(mutations_df)}")
print(f"Columns: {list(mutations_df.columns[:10])}")
mutations_df.head(3)

### 1.3  Building the Binary Mutation Matrix

We create a **samples × genes** binary matrix where 1 = at least one non-synonymous somatic mutation is present.

In [ ]:
# Keep only the columns we need
mut_slim = mutations_df[["sampleId", "gene"]].copy()
mut_slim.columns = ["sample", "gene"]

# Remove duplicates (same gene mutated twice in same sample)
mut_slim = mut_slim.drop_duplicates()

# Pivot to binary matrix (samples × genes)
sm_mat = mut_slim.pivot_table(index="sample", columns="gene", aggfunc=len, fill_value=0)
sm_mat = (sm_mat > 0).astype(int)   # binarise

print(f"Binary mutation matrix shape: {sm_mat.shape}")
print(f"  → {sm_mat.shape[0]} samples, {sm_mat.shape[1]} genes")
print(f"  → Overall mutation rate: {sm_mat.values.mean():.3f}")

# Visualise: top 30 most frequently mutated genes
top_genes = sm_mat.sum().nlargest(30)
plt.figure(figsize=(12, 4))
top_genes.plot(kind="bar", color="steelblue", edgecolor="white")
plt.title(f"Top 30 Most Frequently Mutated Genes — {STUDY_ID}", fontsize=13)
plt.ylabel("Number of samples with mutation")
plt.xlabel("Gene")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("top_mutated_genes.png", dpi=150)
plt.show()

### 1.4  Downloading Clinical / Survival Data

In [ ]:
# Fetch overall survival data
clinical_data = cbio_get(
    "clinical-data/fetch",
)

# Alternative: direct per-study clinical fetch
clin_response = requests.post(
    f"{BASE_URL}/clinical-data/fetch",
    json={"studyIds": [STUDY_ID], "attributeIds": ["OS_STATUS", "OS_MONTHS", "DFS_STATUS", "DFS_MONTHS"]},
    params={"clinicalDataType": "PATIENT", "projection": "SUMMARY"},
    timeout=60
)
clin_df = pd.DataFrame(clin_response.json())

# Pivot so each patient is one row
clin_pivot = clin_df.pivot_table(index="patientId", columns="clinicalAttributeId",
                                  values="value", aggfunc="first")
clin_pivot.columns.name = None
clin_pivot = clin_pivot.reset_index()

print(f"Clinical data shape: {clin_pivot.shape}")
print(f"Columns: {list(clin_pivot.columns)}")
clin_pivot.head(3)

In [ ]:
# Parse survival columns
def parse_survival(df, time_col="OS_MONTHS", status_col="OS_STATUS"):
    """Return a clean survival DataFrame."""
    surv = df[["patientId", time_col, status_col]].copy().dropna()
    surv[time_col] = pd.to_numeric(surv[time_col], errors="coerce")
    # cBioPortal uses '1:DECEASED' / '0:LIVING' format
    surv["event"] = surv[status_col].str.startswith("1").astype(int)
    surv = surv.rename(columns={time_col: "time"})
    surv = surv.dropna(subset=["time"]).query("time >= 0")
    return surv[["patientId", "time", "event"]]

surv_df = parse_survival(clin_pivot)
print(f"Patients with survival data: {len(surv_df)}")
print(f"Events (deaths): {surv_df['event'].sum()} ({surv_df['event'].mean()*100:.1f}%)")
surv_df.head()

---
## Section 2 — Loading the Protein–Protein Interaction Network

We use the **Human Interactome** (STRING / IntAct-based), which provides high-confidence protein interactions. You can download it from:

- **STRING** (recommended): https://string-db.org/cgi/download?species_text=Homo+sapiens  
  → Download `protein.links.v12.0.txt.gz` then filter by `combined_score ≥ 700`
- Or use the preformatted file from pyNBS examples.

> **📌 Note:** For this tutorial we demonstrate with the pyNBS example network. Replace `network_filepath` with your own file.

In [ ]:
# ── Option A: Load your own network file ──────────────────────────────────────
# network_filepath = '/path/to/Human_Interactome_GeneNames_v2.txt'
# G_raw = nx.read_edgelist(network_filepath, data=False)

# ── Option B: Download a minimal STRING edge list on the fly ──────────────────
# (requires ~200 MB download; set DOWNLOAD_STRING = True to use)
DOWNLOAD_STRING = False

if DOWNLOAD_STRING:
    import gzip
    string_url = "https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz"
    print("Downloading STRING network... (this is large, ~200 MB)")
    r = requests.get(string_url, stream=True, timeout=300)
    with gzip.open(io.BytesIO(r.content)) as f:
        string_df = pd.read_csv(f, sep=" ")
    string_high = string_df[string_df["combined_score"] >= 700]
    G_raw = nx.from_pandas_edgelist(string_high, source="protein1", target="protein2")
    print(f"STRING network: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")
else:
    # ── Option C: Simulate a small interactome from known cancer pathways ──────
    print("Using simulated interactome for demonstration.")
    print("For real analysis, use STRING or provide your own network file.")
    # Build a Barabasi–Albert scale-free graph that approximates PPI topology
    # and label nodes with gene names from our mutation matrix
    genes_in_mat = list(sm_mat.columns)
    n_genes = min(len(genes_in_mat), 3000)
    G_raw = nx.barabasi_albert_graph(n_genes, m=3, seed=RANDOM_STATE)
    mapping = {i: genes_in_mat[i] for i in range(n_genes)}
    G_raw = nx.relabel_nodes(G_raw, mapping)
    print(f"Simulated network: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")
  

# Clean node names
G_raw = nx.relabel_nodes(G_raw, {n: str(n).strip() for n in G_raw.nodes()})
print(f"Network after cleaning: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")

### 2.1  Aligning Mutation Matrix and Network

In [ ]:
# Intersection of genes
common_genes = list(set(sm_mat.columns) & set(G_raw.nodes()))
print(f"Genes in mutation matrix:     {sm_mat.shape[1]}")
print(f"Genes in network:             {G_raw.number_of_nodes()}")
print(f"Genes in common:              {len(common_genes)}")

# Filter both to common genes
sm_mat_f = sm_mat[common_genes].copy()
G_f = G_raw.subgraph(common_genes).copy()

# Remove isolated nodes from network
isolated = [n for n, d in G_f.degree() if d == 0]
G_f.remove_nodes_from(isolated)
sm_mat_f = sm_mat_f[[g for g in common_genes if g in G_f.nodes()]]

print(f"\nFinal mutation matrix: {sm_mat_f.shape}")
print(f"Final network:         {G_f.number_of_nodes()} nodes, {G_f.number_of_edges()} edges")

---
## Section 3 — Network Propagation

Network propagation is an **iterative diffusion** process on a graph. Starting from a binary mutation vector $F^{(0)}$ for a patient, we iteratively update:

$$F^{(t+1)} = \alpha \cdot W \cdot F^{(t)} + (1 - \alpha) \cdot F^{(0)}$$

where:
- $W$ is the **column-normalised adjacency matrix** of the PPI network  
- $\alpha \in [0, 1)$ controls **how far** information spreads ($\alpha = 0$: no propagation; $\alpha \to 1$: full diffusion)
- The closed-form solution at convergence is: $F^* = (1-\alpha)(I - \alpha W)^{-1} F^{(0)}$

We implement this efficiently without matrix inversion using iterative updates.


In [ ]:
# Perform network propagation
# Use function from pyNBS: 

# Run network propagation
ALPHA = 0.7   # Standard value from Hofree et al.

print(f"Running network propagation (α = {ALPHA})...")

prop_mat = prop.network_propagation(
    network=G_f,
    binary_matrix=sm_mat_f,
    alpha=ALPHA,
    verbose=True
)

In [ ]:
# or you might have an interest to write the function on your own
# the example realization in Network_propagation.py
def build_normalised_adjacency(G):
    """
    Build the column-normalised adjacency matrix W.
    W_ij = A_ij / deg(j)   — so each column sums to 1.
    """
 

def network_propagation(sm_mat, G, alpha=0.5, tol=1e-6, max_iter=200, verbose=True):
    """
    Propagate binary mutation profiles over a PPI network.

    Parameters
    ----------
    sm_mat  : pd.DataFrame, shape (n_samples, n_genes)
    G       : nx.Graph — must share gene names with sm_mat columns
    alpha   : float, propagation coefficient (0.5 is standard)
    tol     : convergence threshold
    max_iter: maximum iterations

    Returns
    -------
    pd.DataFrame of shape (n_samples, n_network_genes)
    """
    W, net_genes = build_normalised_adjacency(G)

    # Align mutation matrix to network gene order; fill missing genes with 0


    # Normalise each patient's profile (so different mutation burdens don't dominate)



print("✅ Network propagation functions defined.")

In [ ]:
print(f"\nPropagated matrix shape: {prop_mat.shape}")
print(f"Value range: [{prop_mat.values.min():.4f}, {prop_mat.values.max():.4f}]")

### 3.3  Visualising the Effect of Propagation

Compare the sparsity and distribution of raw vs. propagated profiles.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: Raw mutation sparsity
ax = axes[0]
raw_vals = sm_mat_f.values.flatten()
ax.hist(raw_vals, bins=3, color='tomato', edgecolor='white', rwidth=0.7)
ax.set_title('Raw Mutations\n(binary, sparse)', fontsize=11)
ax.set_xlabel('Value'); ax.set_ylabel('Count (log)')
ax.set_yscale('log')

# Panel 2: Propagated profile distribution
ax = axes[1]
prop_vals = prop_mat.values.flatten()
ax.hist(prop_vals[prop_vals > 1e-5], bins=50, color='steelblue', edgecolor='white')
ax.set_title('Propagated Scores\n(continuous, dense)', fontsize=11)
ax.set_xlabel('Propagation score'); ax.set_ylabel('Count')

# Panel 3: Heatmap of top variable genes (raw vs prop)
ax = axes[2]
top_var_genes = prop_mat.var().nlargest(50).index
sample_subset = prop_mat.index[:min(50, len(prop_mat))]
sns.heatmap(
    prop_mat.loc[sample_subset, top_var_genes].T,
    ax=ax, cmap='Blues', xticklabels=False, yticklabels=False,
    cbar_kws={'label': 'Propagation score'}
)
ax.set_title('Propagated profiles\n(samples × genes)', fontsize=11)
ax.set_xlabel('Samples'); ax.set_ylabel('Genes (top 50 variable)')

plt.suptitle(f'Network Propagation Results — {STUDY_ID}', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('propagation_overview.png', dpi=150, bbox_inches='tight')
plt.show()

> **🔍 Question 3.1:** How does the sparsity change after propagation? Why is this beneficial for clustering?

> **🔍 Question 3.2:** Try changing `ALPHA` to 0.1, 0.3, 0.5, and 0.9. How does the propagated profile distribution change? Which α leads to the most 'smoothed' result?

---
## Section 4 — Patient Similarity Network (PSN)

Once we have smooth propagated profiles, we compute **pairwise cosine similarity** between patients. This creates a **Patient Similarity Network** (PSN) that we can cluster.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity (patients × patients)
psn = cosine_similarity(prop_mat.values)
psn_df = pd.DataFrame(psn, index=prop_mat.index, columns=prop_mat.index)

print(f"Patient Similarity Matrix shape: {psn_df.shape}")
print(f"Similarity range: [{psn.min():.3f}, {psn.max():.3f}]")

# Visualise PSN heatmap (sorted by hierarchical clustering)
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list

Z = linkage(1 - psn, method='ward')   # ward on distance = 1 - similarity
order = leaves_list(Z)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(psn[np.ix_(order, order)], cmap='Blues', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_title('Patient Similarity Network (sorted by hierarchy)', fontsize=12)
ax.set_xlabel('Patients'); ax.set_ylabel('Patients')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig('psn_heatmap.png', dpi=150)
plt.show()

---
## Section 5 — Clustering

We will use two complementary approaches:

1. **NMF (Non-negative Matrix Factorisation)** — decomposes the propagated matrix into latent mutation 'signatures' and patient memberships
2. **Hierarchical Consensus Clustering** — assesses cluster stability by repeated subsampling

### 5.1  Selecting the Number of Clusters (k)

Before clustering, we need to decide on $k$. We use:
- **Silhouette score** (higher = better-separated clusters)
- **NMF reconstruction error** (cophenetic correlation)
- **Try randon number e.g. 4 or 5 for educational purposes** that might be easily adaptable

In [ ]:
def nmf_cluster(X, k, n_init=10, random_state=42):
    """
    Run NMF and return cluster labels (argmax of H matrix).
    X : (n_samples, n_features), non-negative
    """
    best_err, best_model = np.inf, None
    for seed in range(n_init):
        model = NMF(n_components=k, init='nndsvda', random_state=random_state + seed,
                    max_iter=500, tol=1e-4)
        W = model.fit_transform(X)
        if model.reconstruction_err_ < best_err:
            best_err = model.reconstruction_err_
            best_model = model
            best_W = W
    H = best_model.components_   # (k, n_features)
    labels = np.argmax(best_W, axis=1)   # patient cluster assignment
    return labels, best_W, H, best_err


# Sweep k from 2 to 7
X = prop_mat.values.copy()
X = np.maximum(X, 0)  # ensure non-negative (cosine-propagated values are already ≥ 0)

k_range = range(2, 8)
sil_scores, recon_errors = [], []

print("Sweeping k (this may take 1–2 minutes)...")
for k in k_range:
    labels, W, H, err = nmf_cluster(X, k, n_init=5)
    sil = silhouette_score(X, labels, metric='cosine') if len(set(labels)) > 1 else 0
    sil_scores.append(sil)
    recon_errors.append(err)
    print(f"  k={k}: silhouette={sil:.3f}, reconstruction error={err:.2f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(k_range), sil_scores, 'o-', color='steelblue', linewidth=2)
ax1.set_xlabel('Number of clusters (k)'); ax1.set_ylabel('Silhouette score')
ax1.set_title('Silhouette Score vs k'); ax1.set_xticks(list(k_range))
ax1.axvline(list(k_range)[np.argmax(sil_scores)], color='red', linestyle='--', label=f'Best k={list(k_range)[np.argmax(sil_scores)]}')
ax1.legend()

ax2.plot(list(k_range), recon_errors, 'o-', color='darkorange', linewidth=2)
ax2.set_xlabel('Number of clusters (k)'); ax2.set_ylabel('NMF reconstruction error')
ax2.set_title('NMF Reconstruction Error vs k'); ax2.set_xticks(list(k_range))

plt.suptitle('Selecting the Number of Clusters', fontsize=13)
plt.tight_layout()
plt.savefig('k_selection.png', dpi=150, bbox_inches='tight')
plt.show()

BEST_K = list(k_range)[np.argmax(sil_scores)]
print(f"\n→ Best k by silhouette: {BEST_K}")

### 5.2  NMF Clustering

In [ ]:
# You can override BEST_K manually if you have a biological reason
K = BEST_K   # or set manually, e.g. K = 3

nmf_labels, W_nmf, H_nmf, _ = nmf_cluster(X, K, n_init=20)

nmf_results = pd.Series(nmf_labels, index=prop_mat.index, name='NMF_cluster')
print(f"NMF cluster sizes (k={K}):")
print(nmf_results.value_counts().sort_index())

# Heatmap of NMF H matrix (clusters × genes)
# Each row = one cluster's 'signature' in gene space
top_genes_per_cluster = []
for i in range(K):
    top = H_nmf[i].argsort()[-10:][::-1]
    top_genes_per_cluster.append(prop_mat.columns[top].tolist())

fig, ax = plt.subplots(figsize=(14, 3))
sig_genes = list(dict.fromkeys([g for cl in top_genes_per_cluster for g in cl]))  # flatten, unique
sig_idx = [list(prop_mat.columns).index(g) for g in sig_genes]
H_sub = H_nmf[:, sig_idx]
sns.heatmap(pd.DataFrame(H_sub, index=[f'Cluster {i}' for i in range(K)], columns=sig_genes),
            cmap='YlOrRd', ax=ax, linewidths=0.5, cbar_kws={'label': 'NMF weight'})
ax.set_title(f'NMF Signature Genes (top 10 per cluster, k={K})', fontsize=12)
ax.set_xlabel('Gene'); ax.set_ylabel('Cluster')
plt.tight_layout()
plt.savefig('nmf_signatures.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3  Hierarchical Consensus Clustering

Consensus clustering evaluates cluster **stability** by repeatedly subsampling patients (80%) and clustering each subsample. The **consensus matrix** records how often each pair of patients lands in the same cluster — stable clusters will have entries close to 0 or 1.

In [ ]:
def consensus_clustering(X, k, n_iter=50, subsample=0.8, random_state=42):
    """
    Hierarchical consensus clustering.

    Parameters
    ----------
    X           : (n_samples, n_features)
    k           : number of clusters
    n_iter      : number of subsampling iterations
    subsample   : fraction of samples to use each iteration

    Returns
    -------
    consensus   : (n_samples, n_samples) matrix, values in [0, 1]
    labels      : final cluster labels from consensus matrix
    """
    n = X.shape[0]
    C = np.zeros((n, n))   # co-occurrence count
    N = np.zeros((n, n))   # co-sampled count

    rng = np.random.RandomState(random_state)

    for it in range(n_iter):
        # Subsample patients
        idx = rng.choice(n, size=int(n * subsample), replace=False)
        X_sub = X[idx]

        # Ward hierarchical clustering on subsampled cosine distance
        model = AgglomerativeClustering(n_clusters=k, linkage='ward')
        sub_labels = model.fit_predict(X_sub)

        # Update co-occurrence and co-sampling matrices
        for ci in range(k):
            members = idx[sub_labels == ci]
            # All pairs in this cluster
            for a in members:
                for b in members:
                    C[a, b] += 1
        # All pairs that were both sampled
        for a in idx:
            for b in idx:
                N[a, b] += 1

    # Consensus matrix
    with np.errstate(invalid='ignore'):
        consensus = np.where(N > 0, C / N, 0)

    # Final clustering on consensus matrix
    dist = 1 - consensus
    np.fill_diagonal(dist, 0)
    final_model = AgglomerativeClustering(n_clusters=k, linkage='average', metric='precomputed')
    labels = final_model.fit_predict(dist)

    return consensus, labels


print(f"Running consensus clustering (k={K}, 50 iterations)...")
print("This may take a few minutes for large cohorts.")

consensus_mat, hc_labels = consensus_clustering(X, k=K, n_iter=50, subsample=0.8)

hc_results = pd.Series(hc_labels, index=prop_mat.index, name='HC_cluster')
print(f"\nHierarchical consensus cluster sizes (k={K}):")
print(hc_results.value_counts().sort_index())

In [ ]:
# Visualise consensus matrix
# Sort patients by cluster assignment
order_hc = np.argsort(hc_labels)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(consensus_mat[np.ix_(order_hc, order_hc)],
               cmap='Blues', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Consensus index')
ax.set_title(f'Consensus Matrix (k={K}, sorted by cluster)', fontsize=12)
ax.set_xlabel('Patients'); ax.set_ylabel('Patients')
ax.set_xticks([]); ax.set_yticks([])

# Draw cluster boundaries
boundaries = []
for cl in range(K):
    n_in_cl = (hc_labels[order_hc] == cl).sum()
    boundaries.append(boundaries[-1] + n_in_cl if boundaries else n_in_cl)
for b in boundaries[:-1]:
    ax.axhline(b - 0.5, color='red', linewidth=1.5)
    ax.axvline(b - 0.5, color='red', linewidth=1.5)

plt.tight_layout()
plt.savefig('consensus_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Interpretation:")
print("  Dark blue blocks along the diagonal → stable clusters (patients always cluster together)")
print("  Light off-diagonal values → patients rarely co-cluster → good separation")

### 5.4  Comparing NMF vs Hierarchical Consensus Clustering

In [ ]:
from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(nmf_labels, hc_labels)
print(f"Adjusted Rand Index (NMF vs HC): {ari:.3f}")
print("  ARI = 1.0 → perfect agreement; ARI ~ 0 → random agreement")

# Crosstab
comparison = pd.crosstab(
    pd.Series(nmf_labels, name='NMF cluster'),
    pd.Series(hc_labels, name='HC cluster')
)
print("\nCrosstab of NMF vs HC cluster assignments:")
comparison

> **🔍 Question 5.1:** Are the two clustering methods in agreement? What does a high ARI tell you?

> **🔍 Question 5.2:** Look at the consensus matrix. Are all clusters equally stable, or are some less well-defined?

---
## Section 6 — Kaplan–Meier Survival Analysis

The key clinical question: **do the identified patient subgroups have different survival outcomes?**

We use **Kaplan–Meier curves** and the **log-rank test** to assess this.

In [ ]:
# Map cluster labels to patient IDs
# cBioPortal sample IDs often follow pattern TCGA-XX-XXXX-01 → patient is TCGA-XX-XXXX
def sample_to_patient(s):
    parts = s.split('-')
    return '-'.join(parts[:3]) if len(parts) >= 3 else s

cluster_df = pd.DataFrame({
    'sampleId': prop_mat.index,
    'patientId': [sample_to_patient(s) for s in prop_mat.index],
    'NMF_cluster': nmf_labels,
    'HC_cluster': hc_labels
})

# Merge with survival data
surv_clusters = surv_df.merge(cluster_df, on='patientId', how='inner')
print(f"Patients with both survival and cluster data: {len(surv_clusters)}")
surv_clusters.head()

In [ ]:
def plot_km(surv_df, cluster_col, title, ax, palette=None):
    """
    Plot Kaplan–Meier curves for each cluster.
    Returns log-rank p-value (multivariate).
    """
    kmf = KaplanMeierFitter()
    clusters = sorted(surv_df[cluster_col].unique())
    colors = palette or cm.tab10.colors

    for i, cl in enumerate(clusters):
        mask = surv_df[cluster_col] == cl
        T = surv_df.loc[mask, 'time']
        E = surv_df.loc[mask, 'event']
        kmf.fit(T, event_observed=E, label=f'Cluster {cl} (n={mask.sum()})')
        kmf.plot_survival_function(ax=ax, color=colors[i % len(colors)],
                                    ci_show=True, ci_alpha=0.1)

    # Log-rank test (multivariate)
    result = multivariate_logrank_test(
        surv_df['time'], surv_df[cluster_col], surv_df['event']
    )
    pval = result.p_value

    ax.set_xlabel('Time (months)'); ax.set_ylabel('Survival probability')
    ax.set_title(f"{title}\nlog-rank p = {pval:.4f}", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.legend(loc='lower left', fontsize=8)
    return pval


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(surv_clusters) > 10:
    p_nmf = plot_km(surv_clusters, 'NMF_cluster', f'NMF Clustering (k={K})', axes[0])
    p_hc  = plot_km(surv_clusters, 'HC_cluster',  f'Hierarchical Consensus (k={K})', axes[1])
    plt.suptitle(f'Kaplan–Meier Survival by Patient Subgroup — {STUDY_ID}', fontsize=13)
    plt.tight_layout()
    plt.savefig('km_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nNMF log-rank p-value:        {p_nmf:.4f}")
    print(f"HC  log-rank p-value:        {p_hc:.4f}")
    if p_nmf < 0.05 or p_hc < 0.05:
        print("✅ Significant survival difference between clusters (p < 0.05)!")
    else:
        print("⚠️  No significant survival difference — try a different k or check data quality.")
else:
    print("Not enough matched patients for survival analysis. Check patient ID mapping.")

### 6.2  Pairwise Log-Rank Tests

In [ ]:
# Pairwise log-rank tests between clusters (using NMF labels)
from itertools import combinations

clusters = sorted(surv_clusters['NMF_cluster'].unique())
print(f"Pairwise log-rank tests (NMF clustering, k={K}):")
print(f"{'Cluster A':>12} {'Cluster B':>12} {'p-value':>12} {'Significant':>12}")
print("-" * 52)

for ca, cb in combinations(clusters, 2):
    mask = surv_clusters['NMF_cluster'].isin([ca, cb])
    sub = surv_clusters[mask]
    res = logrank_test(
        sub.loc[sub['NMF_cluster'] == ca, 'time'],
        sub.loc[sub['NMF_cluster'] == cb, 'time'],
        event_observed_A=sub.loc[sub['NMF_cluster'] == ca, 'event'],
        event_observed_B=sub.loc[sub['NMF_cluster'] == cb, 'event']
    )
    sig = '✅' if res.p_value < 0.05 else ''
    print(f"{ca:>12} {cb:>12} {res.p_value:>12.4f} {sig:>12}")

> **🔍 Question 6.1:** Which pair of clusters has the most significant survival difference? Does this make biological sense?

> **🔍 Question 6.2:** Repeat the KM analysis using the HC clusters. Does the clustering method affect the clinical significance?

---
## Section 7 — Characterising the Subtypes

Now that we have subtypes, we want to understand what biological features define them.

### 7.1  Top Mutated Genes per Cluster

In [ ]:
# Merge raw mutation data with cluster labels
mut_clusters = sm_mat_f.copy()
mut_clusters.index.name = 'sampleId'
mut_clusters = mut_clusters.reset_index()
mut_clusters = mut_clusters.merge(
    cluster_df[['sampleId', 'NMF_cluster']], on='sampleId', how='inner'
).set_index('sampleId')

# Mutation frequency per cluster
cluster_mut_freq = mut_clusters.groupby('NMF_cluster').mean().T  # genes × clusters
cluster_mut_freq.columns = [f'Cluster {c}' for c in cluster_mut_freq.columns]

# Select top differentially mutated genes
cluster_mut_freq['max_freq'] = cluster_mut_freq.max(axis=1)
cluster_mut_freq['range'] = cluster_mut_freq.max(axis=1) - cluster_mut_freq.min(axis=1)
top_diff = cluster_mut_freq.nlargest(20, 'range').drop(columns=['max_freq', 'range'])

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(top_diff * 100, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=ax, cbar_kws={'label': 'Mutation frequency (%)'},
            linewidths=0.5)
ax.set_title(f'Top 20 Differentially Mutated Genes (k={K})', fontsize=12)
ax.set_xlabel('Cluster'); ax.set_ylabel('Gene')
plt.tight_layout()
plt.savefig('diff_mutated_genes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary & Discussion

In this tutorial you have:

1. **Downloaded** real cancer genomics data from cBioPortal using the REST API
2. **Built** a binary somatic mutation matrix
3. **Applied network propagation** to smooth sparse mutation profiles over a PPI network
4. **Computed** a Patient Similarity Network via cosine similarity
5. **Clustered** patients using NMF and Hierarchical Consensus Clustering
6. **Evaluated** cluster stability and selected the optimal number of subtypes
7. **Assessed** clinical relevance with Kaplan–Meier survival analysis and log-rank tests
8. **Characterised** subtypes by differentially mutated genes

---

### Exercises 

1. **Different dataset:** Repeat the full analysis with a different cBioPortal study. Do you observe different numbers of stable clusters?

2. **Alpha sensitivity:** Run propagation with α ∈ {0.1, 0.3, 0.5, 0.7, 0.9}. For each α, compute the silhouette score and KM log-rank p-value. Plot these against α and discuss the optimal trade-off.

3. **No propagation baseline:** Cluster patients directly from the raw binary mutation matrix (skipping Section 3). Compare the KM log-rank p-value to your propagated result. Does network propagation improve clinical stratification?

4. **Network choice:** Replace the interactome with STRING at a different confidence threshold (e.g., ≥ 500, ≥ 900). How does network density affect the propagated profiles and final clustering?

5. **Biological annotation:** For your best clustering result, look up the top mutated genes in each cluster on databases such as COSMIC or KEGG. Propose a biological interpretation for each subtype.

6. **Input File Filtering:** try CADD/SIFT/PolyPhen filtering for damaging mutations only.

7. **Construction of Patient Similarity Networks (PSNs) from smoothed mutation profiles:** several alternatives to cosine similarity exist, e.g. Pearson Correlation Euclidean/Manhattan Distance.




---

### References

- Hofree M. et al. (2013) *Network-based stratification of tumour mutations.* **Nature Methods** 10, 1108–1115.
- Vandin F. et al. (2011) *Algorithms for detecting significantly mutated pathways in cancer.* **J Comp Biol** 18, 507–522.
- Monti S. et al. (2003) *Consensus Clustering.* **Machine Learning** 52, 91–118.
- Cerami E. et al. (2012) *The cBio Cancer Genomics Portal.* **Cancer Discovery** 2, 401–404.